In [0]:
# ============================================================
# Bronze — Source 18: AWS SES Email
#
# Incremental load using watermark pattern.
#
# Source: s3://ecommerce-lakehouse-467091806172-raw-01/source=18_ses_email/
# Target: bronze.ses catalog in Unity Catalog
#
# Tables:
#   - bronze.ses.email_events
#
# Raw format: JSON (hourly partitioned)
# ============================================================

import sys
sys.path.append("/Workspace/Users/sutharripal26@gmail.com/ecommerce-lakehouse/pipelines/bronze/shared")
from bronze_utils import get_watermark, update_watermark
from pyspark.sql.functions import col, lit, max as spark_max

spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

RAW_BUCKET = "s3://ecommerce-lakehouse-467091806172-raw-01"
SOURCE = "18_ses_email"

dbutils.fs.ls(f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=17/")

In [0]:
# Cell 2 — Read all SES email event JSON files
watermark = get_watermark(spark, SOURCE)
print(f"Loading files modified after: {watermark}")

path = f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=*/hour=*/ses_events_*.json"

df = spark.read.format("json") \
    .load(path) \
    .filter(col("_metadata.file_modification_time") > lit(watermark))

print(f"Raw rows: {df.count()}")
df.printSchema()

In [0]:
# Cell 3 — Flatten SES email events with correct fields
from pyspark.sql.functions import explode

emails_flat = df \
    .select(explode(col("events")).alias("e")) \
    .select(
        col("e.message_id"),
        col("e.event_type"),
        col("e.sent_at"),
        col("e.delivered_at"),
        col("e.delivery_status"),
        col("e.customer_id"),
        col("e.order_id"),
        col("e.recipient"),
        col("e.sender"),
        col("e.subject"),
        col("e.ses_message_id"),
        col("e.ses_region"),
        col("e.sending_account_id"),
        col("e.configuration_set"),
        col("e.bounce.bounce_type").alias("bounce_type"),
        col("e.bounce.bounce_subtype").alias("bounce_subtype"),
        col("e.bounce.bounced_at").alias("bounced_at"),
        col("e.bounce.status").alias("bounce_status"),
        col("e.complaint.complaint_type").alias("complaint_type"),
        col("e.complaint.complained_at").alias("complained_at"),
        col("e.complaint.feedback_id").alias("complaint_feedback_id"),
        col("e.engagement.clicked").alias("clicked"),
        col("e.engagement.opened").alias("opened"),
        col("e.engagement.unsubscribed").alias("unsubscribed"),
    )

print(f"Flattened emails: {emails_flat.count()} rows")
emails_flat.show(3)

In [0]:
# Cell 4 — Write to Bronze and update watermark
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze.ses")

emails_flat.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze.ses.email_events")

latest_ts = df.select(spark_max("_metadata.file_modification_time")).collect()[0][0]
update_watermark(spark, SOURCE, latest_ts, emails_flat.count())
print(f"✅ bronze.ses.email_events: {emails_flat.count()} rows")

In [0]:
count = spark.sql("SELECT COUNT(*) as cnt FROM bronze.ses.email_events").collect()[0]['cnt']
print(f"bronze.ses.email_events: {count} rows")
spark.sql("SELECT * FROM bronze.pipeline.watermarks WHERE source = '18_ses_email'").show()